# N1-AT1 | Sistema de Auditoria e Analise de Dados Reais em Python
## Habitos e rotinas de estudo de universitarios
Turma: Bacharelado em Ciencias da Computacao - Periodo Noturno - Novas Tecnologias - GPE02N50131

Integrantes: Paulo José Higa Freitas, Pedro Henrique Fernandes, Vinicius Velasquez Coelho

Tema escolhido: habitos e rotinas de estudo de universitarios.
Descricao do problema: O sistema coleta respostas de um formulario em CSV, identifica inconsistencias, padroniza os registros e gera rankings, filtros, buscas por termo e analises para apoiar a interpretacao da base.


# EXPLICACAO DA BASE DE DADOS
Base de dados construida pelo grupo:
- Total de registros usados nos testes atuais: 42.
- Total validado na base oficial atual: 35 registros validos, 7 invalidos e 0 duplicados.
- Meta da entrega final: pelo menos 40 registros validos.
- Campos:
  * timestamp
  * idade
  * area_curso
  * semestre
  * turno
  * horas_sono
  * horas_estudo
  * estuda_fds
  * lugar_estudo
  * produtividade
  * bebe_cafe
  * horas_redes
  * trabalha
  * dificuldade

A base final deve continuar anonima.


# REGRAS DE VALIDACAO
Regras de validacao implementadas:
1. Idade deve estar entre 18 e 80 anos.
2. Semestre deve estar entre 1 e 12.
3. Horas de sono devem estar entre 1 e 24.
4. Horas de estudo devem estar entre 0 e 18.
5. Horas em redes sociais devem estar entre 0 e 12.
6. Campos obrigatorios nao podem estar vazios.
7. Turno deve ser: matutino, vespertino, noturno ou integral.
8. Produtividade deve ser: muito bom, bom, normal, ruim ou muito ruim.
9. Campos booleanos devem ser respondidos como sim ou nao.
10. Registros duplicados sao invalidos.
11. A soma de horas de sono, estudo e redes nao pode ultrapassar 24 horas no mesmo registro.

# IMPLEMENTACAO DO SISTEMA


### Bibliotecas utilizadas
Esta celula importa apenas recursos compativeis com o trabalho: upload de arquivo no Colab e leitura/escrita de CSV.


In [1]:
from google.colab import files
import csv

ModuleNotFoundError: No module named 'google'

### Upload do arquivo
Esta celula recebe o arquivo CSV enviado no Google Colab para que a base possa ser carregada no sistema.


In [ ]:
uploaded = files.upload()
nome_arquivo = list(uploaded.keys())[0]
print(f'\nArquivo carregado: {nome_arquivo}')

Saving dados_habitos_oficial - Respostas ao formulário 4.csv to dados_habitos_oficial - Respostas ao formulário 4.csv

Arquivo carregado: dados_habitos_oficial - Respostas ao formulário 4.csv


## Funcoes do sistema
A seguir, cada bloco de codigo aparece com o nome das funcoes e uma explicacao curta da sua finalidade.


### `carregar_arquivo()`
Serve para ler o CSV, transformar cada linha em um dicionario e registrar linhas malformadas como invalidas para a auditoria.


In [ ]:
def carregar_arquivo(nome_arquivo):
    """
    Funcao para carregar o arquivo CSV e retornar uma lista de dicionarios.
    Linhas quebradas entram como invalidas para nao sumirem da auditoria.
    """
    dados = []
    campos_esperados = [
        'timestamp', 'idade', 'area_curso', 'semestre', 'turno',
        'horas_sono', 'horas_estudo', 'estuda_fds', 'lugar_estudo',
        'produtividade', 'bebe_cafe', 'horas_redes', 'trabalha', 'dificuldade'
    ]

    try:
        with open(nome_arquivo, 'r', encoding='utf-8-sig', newline='') as arquivo:
            leitor = csv.reader(arquivo)
            next(leitor, None)

            for numero_linha, campos in enumerate(leitor, start=2):
                if len(campos) != len(campos_esperados):
                    registro_invalido = {
                        campo: campos[indice] if indice < len(campos) else ''
                        for indice, campo in enumerate(campos_esperados)
                    }
                    registro_invalido['status_validacao'] = 'invalido'
                    registro_invalido['erros_validacao'] = [
                        f"Linha CSV malformada na linha {numero_linha}: esperadas {len(campos_esperados)} colunas e encontradas {len(campos)}."
                    ]
                    dados.append(registro_invalido)
                    continue

                registro = dict(zip(campos_esperados, campos))
                dados.append(registro)

    except FileNotFoundError:
        print(f"Erro: Arquivo '{nome_arquivo}' nao encontrado!")
        return []

    return dados


### `converter_numero()`
Serve para converter valores numericos e horarios em formato de texto para numeros usados na validacao, filtros e estatisticas.


In [ ]:
def converter_numero(valor, tipo='float'):
    """
    Converte strings numericas ou horarios HH:MM ou HH:MM:SS para numero.
    Retorna None quando a conversao nao for possivel.
    """
    if valor is None:
        return None

    texto = str(valor).strip().replace(',', '.')
    if texto == '':
        return None

    try:
        if ':' in texto:
            partes = texto.split(':')
            if len(partes) not in [2, 3]:
                return None
            horas = float(partes[0])
            minutos = float(partes[1])
            segundos = float(partes[2]) if len(partes) == 3 else 0
            if minutos < 0 or minutos >= 60 or segundos < 0 or segundos >= 60:
                return None
            numero = horas + (minutos / 60) + (segundos / 3600)
        else:
            numero = float(texto)

        if tipo == 'int':
            if numero != int(numero):
                return None
            return int(numero)
        return float(numero)
    except ValueError:
        return None


### `normalizar_texto()`
Serve para padronizar textos, removendo espacos extras, letras maiusculas e acentos mais comuns.


In [ ]:
def normalizar_texto(texto):
    """
    Padroniza texto removendo espacos extras e acentos mais comuns.
    """
    if texto is None:
        return ''

    texto = ' '.join(str(texto).strip().lower().split())
    texto = texto.replace('á', 'a').replace('à', 'a').replace('â', 'a').replace('ã', 'a')
    texto = texto.replace('é', 'e').replace('ê', 'e')
    texto = texto.replace('í', 'i')
    texto = texto.replace('ó', 'o').replace('ô', 'o').replace('õ', 'o')
    texto = texto.replace('ú', 'u')
    texto = texto.replace('ç', 'c')
    return texto

### `normalizar_hora()`
- `normalizar_hora()` converte horarios para valor numerico.



In [ ]:
def normalizar_hora(hora_str):
    """
    Converte valores de hora para numero decimal.
    """
    valor = converter_numero(hora_str, 'float')
    return valor if valor is not None else 0.0

### `formatar_horas()`
- `formatar_horas()` transforma horas decimais em `HH:MM` para exibicao.


In [ ]:
def formatar_horas(valor):
    """
    Converte horas decimais para o formato HH:MM.
    """
    if valor is None:
        return '00:00'

    total_minutos = int(round(float(valor) * 60))
    horas = total_minutos // 60
    minutos = total_minutos % 60
    return f"{horas:02d}:{minutos:02d}"

### `remover_duplicados()`
- `remover_duplicados()` separa registros unicos de registros repetidos.

In [ ]:
def remover_duplicados(dados):
    """
    Separa registros unicos de registros duplicados usando set e tuplas.
    """
    campos_chave = [
        'idade', 'area_curso', 'semestre', 'turno', 'horas_sono',
        'horas_estudo', 'estuda_fds', 'lugar_estudo', 'produtividade',
        'bebe_cafe', 'horas_redes', 'trabalha', 'dificuldade'
    ]

    assinaturas = set()
    dados_unicos = []
    dados_duplicados = []

    for registro in dados:
        assinatura = tuple(normalizar_texto(registro.get(campo, '')) for campo in campos_chave)

        if assinatura in assinaturas:
            registro_duplicado = dict(registro)
            registro_duplicado['status_validacao'] = 'duplicado'
            registro_duplicado['erros_validacao'] = ['Registro duplicado identificado pela assinatura dos campos principais.']
            dados_duplicados.append(registro_duplicado)
        else:
            assinaturas.add(assinatura)
            dados_unicos.append(registro)

    return dados_unicos, dados_duplicados

### `validar_registro()`
Serve para aplicar todas as regras de auditoria e retornar se o registro e valido, junto com os erros encontrados.


In [ ]:
def validar_registro(registro):
    """
    Valida um registro de acordo com as regras definidas.
    Retorna uma tupla com status booleano e lista de erros.
    """
    erros = []
    campos_obrigatorios = [
        'timestamp', 'idade', 'area_curso', 'semestre', 'turno',
        'horas_sono', 'horas_estudo', 'estuda_fds', 'lugar_estudo',
        'produtividade', 'bebe_cafe', 'horas_redes', 'trabalha', 'dificuldade'
    ]

    for campo in campos_obrigatorios:
        if normalizar_texto(registro.get(campo, '')) == '':
            erros.append(f"Campo obrigatorio vazio: {campo}")

    idade = converter_numero(registro.get('idade'), 'int')
    if idade is None or idade < 18 or idade > 80:
        erros.append(f"Idade invalida: {registro.get('idade', '')}")

    semestre = converter_numero(registro.get('semestre'), 'int')
    if semestre is None or semestre < 1 or semestre > 12:
        erros.append(f"Semestre invalido: {registro.get('semestre', '')}")

    turno_valido = {'matutino', 'vespertino', 'noturno', 'integral'}
    if normalizar_texto(registro.get('turno', '')) not in turno_valido:
        erros.append(f"Turno invalido: {registro.get('turno', '')}")

    produtividade_valida = {'muito bom', 'bom', 'normal', 'ruim', 'muito ruim'}
    if normalizar_texto(registro.get('produtividade', '')) not in produtividade_valida:
        erros.append(f"Produtividade invalida: {registro.get('produtividade', '')}")

    respostas_booleanas = {'sim', 's', 'nao', 'n'}
    for campo in ['estuda_fds', 'bebe_cafe', 'trabalha']:
        if normalizar_texto(registro.get(campo, '')) not in respostas_booleanas:
            erros.append(f"Valor booleano invalido em {campo}: {registro.get(campo, '')}")

    horas_sono = converter_numero(registro.get('horas_sono'), 'float')
    horas_estudo = converter_numero(registro.get('horas_estudo'), 'float')
    horas_redes = converter_numero(registro.get('horas_redes'), 'float')

    if horas_sono is None or horas_sono < 1 or horas_sono > 24:
        erros.append(f"Horas de sono invalidas: {registro.get('horas_sono', '')}")

    if horas_estudo is None or horas_estudo < 0 or horas_estudo > 18:
        erros.append(f"Horas de estudo invalidas: {registro.get('horas_estudo', '')}")

    if horas_redes is None or horas_redes < 0 or horas_redes > 12:
        erros.append(f"Horas em redes sociais invalidas: {registro.get('horas_redes', '')}")

    if horas_sono is not None and horas_estudo is not None and horas_redes is not None and (horas_sono + horas_estudo + horas_redes) > 24:
        erros.append('Inconsistencia detectada: soma de sono, estudo e redes acima de 24 horas.')

    return len(erros) == 0, erros

### `filtrar_registros()`
- `filtrar_registros()` realiza filtros textuais por correspondencia exata ou parcial.



In [ ]:
def filtrar_registros(dados, campo, valor, exato=False):
    """
    Filtra registros textuais por igualdade ou ocorrencia parcial.
    Usa filter para destacar o criterio de selecao.
    """
    valor_busca = normalizar_texto(valor)
    if not valor_busca:
        return []

    def corresponde(registro):
        valor_registro = normalizar_texto(registro.get(campo, ''))
        return valor_registro == valor_busca if exato else valor_busca in valor_registro

    return list(filter(corresponde, dados))

### `filtrar_por_intervalo()`
- `filtrar_por_intervalo()` realiza filtros numericos por faixa de valores.


In [ ]:
def filtrar_por_intervalo(dados, campo, minimo=None, maximo=None):
    """
    Filtra registros numericos por intervalo.
    """
    limite_minimo = converter_numero(minimo, 'float') if minimo not in [None, ''] else None
    limite_maximo = converter_numero(maximo, 'float') if maximo not in [None, ''] else None

    def dentro_do_intervalo(registro):
        valor = converter_numero(registro.get(campo), 'float')
        if valor is None:
            return False
        if limite_minimo is not None and valor < limite_minimo:
            return False
        if limite_maximo is not None and valor > limite_maximo:
            return False
        return True

    return list(filter(dentro_do_intervalo, dados))

### `buscar_por_termo()`
Serve para procurar uma palavra ou expressao em um campo especifico ou em todo o registro.


In [ ]:
def buscar_por_termo(dados, termo, campos=None):
    """
    Busca um termo em campos especificos ou em todos os campos do registro.
    """
    if not dados:
        return []

    termo_busca = normalizar_texto(termo)
    if not termo_busca:
        return []

    if campos is None:
        campos = list(dados[0].keys())

    resultados = []
    for registro in dados:
        for campo in campos:
            if termo_busca in normalizar_texto(registro.get(campo, '')):
                resultados.append(registro)
                break

    return resultados

### Funcoes auxiliares de estatistica e analise
- `media()`, `mediana()` e `moda()` calculam medidas basicas.
- `percentual_condicional()`, `media_filtrada()` e `medias_por_grupo()` calculam percentuais, comparacoes e agrupamentos.
- `extrair_series_numericas()`, `calcular_estatisticas_basicas()`, `calcular_percentuais()`, `calcular_medias_comparativas()` e `calcular_indicadores_por_grupo()` organizam os calculos.


In [ ]:
def media(lista):
    """
    Calcula a media de uma lista numerica.
    """
    return sum(lista) / len(lista) if lista else 0

def mediana(lista):
    """
    Calcula a mediana de uma lista numerica.
    """
    if not lista:
        return 0

    lista_ordenada = sorted(lista)
    meio = len(lista_ordenada) // 2
    if len(lista_ordenada) % 2 == 0:
        return (lista_ordenada[meio - 1] + lista_ordenada[meio]) / 2
    return lista_ordenada[meio]

def moda(lista):
    """
    Retorna o valor mais frequente de uma lista.
    """
    if not lista:
        return 'nao identificado'

    frequencias = {}
    for item in lista:
        frequencias[item] = frequencias.get(item, 0) + 1
    return max(frequencias.items(), key=lambda item: item[1])[0]

def percentual_condicional(dados, condicao):
    """
    Calcula o percentual de registros que atendem a uma condicao.
    """
    total = len(dados)
    positivos = sum(1 for registro in dados if condicao(registro))
    return (positivos / total) * 100 if total else 0

def media_filtrada(dados, condicao, campo):
    """
    Calcula a media de um campo apos aplicar um filtro.
    """
    valores = [
        normalizar_hora(registro.get(campo))
        for registro in dados
        if condicao(registro)
    ]
    return media(valores)

def medias_por_grupo(dados, campo_grupo, campo_valor):
    """
    Calcula medias de um campo numerico agrupadas por um campo textual.
    """
    grupos = {}
    for registro in dados:
        grupo = normalizar_texto(registro.get(campo_grupo, ''))
        valor = converter_numero(registro.get(campo_valor), 'float')
        if grupo and valor is not None:
            grupos.setdefault(grupo, []).append(valor)
    return {grupo: media(valores) for grupo, valores in grupos.items()}

def extrair_series_numericas(dados):
    """
    Organiza as principais series numericas usadas nas estatisticas.
    """
    return {
        'idades': list(filter(lambda valor: valor is not None, map(lambda reg: converter_numero(reg.get('idade'), 'int'), dados))),
        'semestres': list(filter(lambda valor: valor is not None, map(lambda reg: converter_numero(reg.get('semestre'), 'int'), dados))),
        'horas_sono': list(map(lambda reg: normalizar_hora(reg.get('horas_sono')), dados)),
        'horas_estudo': list(map(lambda reg: normalizar_hora(reg.get('horas_estudo')), dados)),
        'horas_redes': list(map(lambda reg: normalizar_hora(reg.get('horas_redes')), dados))
    }


def calcular_estatisticas_basicas(dados, series):
    """
    Reune medidas gerais da base validada.
    """
    areas_unicas = set(filter(None, map(lambda reg: normalizar_texto(reg.get('area_curso', '')), dados)))
    total_areas = len(areas_unicas)

    return {
        'total_registros': len(dados),
        'media_idade': media(series['idades']),
        'media_semestre': media(series['semestres']),
        'media_horas_sono': media(series['horas_sono']),
        'media_horas_estudo': media(series['horas_estudo']),
        'media_horas_redes': media(series['horas_redes']),
        'mediana_horas_estudo': mediana(series['horas_estudo']),
        'max_horas_estudo': max(series['horas_estudo']) if series['horas_estudo'] else 0,
        'min_horas_sono': min(series['horas_sono']) if series['horas_sono'] else 0,
        'moda_produtividade': moda(list(map(lambda reg: normalizar_texto(reg.get('produtividade', '')), dados))),
        'moda_turno': moda(list(map(lambda reg: normalizar_texto(reg.get('turno', '')), dados))),
        'total_areas': total_areas,
        'total_areas_unicas': total_areas,
        'atingiu_minimo_registros': len(dados) >= 40
    }


def calcular_percentuais(dados):
    """
    Calcula percentuais de habitos e perfis da base.
    """
    return {
        'percentual_estuda_fds': percentual_condicional(dados, lambda registro: normalizar_texto(registro.get('estuda_fds', '')) in {'sim', 's'}),
        'percentual_trabalham': percentual_condicional(dados, lambda registro: normalizar_texto(registro.get('trabalha', '')) in {'sim', 's'}),
        'percentual_bebem_cafe': percentual_condicional(dados, lambda registro: normalizar_texto(registro.get('bebe_cafe', '')) in {'sim', 's'}),
        'percentual_produtividade_positiva': percentual_condicional(dados, lambda registro: normalizar_texto(registro.get('produtividade', '')) in {'bom', 'muito bom'}),
        'percentual_sono_abaixo_6h': percentual_condicional(dados, lambda registro: normalizar_hora(registro.get('horas_sono')) < 6),
        'percentual_redes_acima_4h': percentual_condicional(dados, lambda registro: normalizar_hora(registro.get('horas_redes')) >= 4)
    }


def calcular_medias_comparativas(dados, series):
    """
    Compara medias entre grupos relacionados ao estudo.
    """
    media_redes = media(series['horas_redes'])

    return {
        'media_estudo_quem_trabalha': media_filtrada(dados, lambda registro: normalizar_texto(registro.get('trabalha', '')) in {'sim', 's'}, 'horas_estudo'),
        'media_estudo_quem_nao_trabalha': media_filtrada(dados, lambda registro: normalizar_texto(registro.get('trabalha', '')) in {'nao', 'n'}, 'horas_estudo'),
        'media_estudo_fim_semana': media_filtrada(dados, lambda registro: normalizar_texto(registro.get('estuda_fds', '')) in {'sim', 's'}, 'horas_estudo'),
        'razao_estudo_redes': media(series['horas_estudo']) / media_redes if media_redes else 0
    }


def calcular_indicadores_por_grupo(dados):
    """
    Identifica destaques por turno e area.
    """
    media_estudo_por_turno = medias_por_grupo(dados, 'turno', 'horas_estudo')
    media_estudo_por_area = medias_por_grupo(dados, 'area_curso', 'horas_estudo')

    turno_maior_media_estudo = max(media_estudo_por_turno, key=media_estudo_por_turno.get) if media_estudo_por_turno else 'nao identificado'
    area_maior_media_estudo = max(media_estudo_por_area, key=media_estudo_por_area.get) if media_estudo_por_area else 'nao identificado'

    return {
        'turno_maior_media_estudo': turno_maior_media_estudo,
        'valor_turno_maior_media_estudo': media_estudo_por_turno.get(turno_maior_media_estudo, 0),
        'area_maior_media_estudo': area_maior_media_estudo,
        'valor_area_maior_media_estudo': media_estudo_por_area.get(area_maior_media_estudo, 0)
    }


### `gerar_estatisticas()` e `gerar_analises_textuais()`
- montam os resultados finais da analise.

In [ ]:
def gerar_estatisticas(dados):
    """
    Gera estatisticas numericas e indicadores personalizados.
    """
    if not dados:
        return {}

    series = extrair_series_numericas(dados)
    estatisticas = {}
    estatisticas.update(calcular_estatisticas_basicas(dados, series))
    estatisticas.update(calcular_percentuais(dados))
    estatisticas.update(calcular_medias_comparativas(dados, series))
    estatisticas.update(calcular_indicadores_por_grupo(dados))
    return estatisticas


def gerar_analises_textuais(dados):
    """
    Gera analises textuais para apoiar a interpretacao da base.
    """
    if not dados:
        return {}

    def moda_textual(valores):
        frequencias = {}
        for valor in valores:
            frequencias[valor] = frequencias.get(valor, 0) + 1
        return max(frequencias.items(), key=lambda item: item[1])[0] if frequencias else 'nao identificado'

    areas = list(filter(None, map(lambda reg: normalizar_texto(reg.get('area_curso', '')), dados)))
    dificuldades = list(filter(None, map(lambda reg: normalizar_texto(reg.get('dificuldade', '')), dados)))
    locais = []
    for registro in dados:
        resposta_local = registro.get('lugar_estudo', '')
        for local in str(resposta_local).split(','):
            local_normalizado = normalizar_texto(local)
            if local_normalizado:
                locais.append(local_normalizado)

    areas_unicas = sorted(set(areas))

    return {
        'area_mais_frequente': moda_textual(areas),
        'dificuldade_mais_citada': moda_textual(dificuldades),
        'local_estudo_mais_comum': moda_textual(locais),
        'areas_identificadas': ', '.join(areas_unicas) if areas_unicas else 'nenhuma'
    }

### `gerar_ranking()`
Serve para ordenar os registros de acordo com um campo numerico e montar rankings de estudo e redes sociais.


In [ ]:
def gerar_ranking(dados, campo, ordem='desc'):
    """
    Gera um ranking baseado em um campo numerico.
    Retorna uma lista de tuplas (area_curso, turno, valor).
    """
    if not dados:
        return []

    itens = []
    for registro in dados:
        valor = converter_numero(registro.get(campo), 'float')
        if valor is not None:
            itens.append((registro.get('area_curso', 'N/A'), registro.get('turno', 'N/A'), valor))

    itens.sort(key=lambda item: item[2], reverse=(ordem == 'desc'))
    return itens

### `salvar_relatorio()` e `salvar_csv()`
- `salvar_relatorio()` gera um arquivo texto com estatisticas, analises, rankings e erros de validacao.
- `salvar_csv()` exporta os registros validos e invalidos para arquivos CSV.


In [ ]:
def salvar_relatorio(dados_validos, dados_invalidos, estatisticas, analises_textuais, ranking_estudo, ranking_redes, nome_arquivo):
    """
    Salva um relatorio completo em arquivo texto.
    """
    with open(nome_arquivo, 'w', encoding='utf-8') as arquivo:
        arquivo.write('=' * 80 + '\n')
        arquivo.write('RELATORIO DE ANALISE - HABITOS DE ESTUDO\n')
        arquivo.write('=' * 80 + '\n\n')

        arquivo.write('ESTATISTICAS GERAIS:\n')
        arquivo.write('-' * 40 + '\n')
        arquivo.write(f"Total de registros validos: {estatisticas.get('total_registros', 0)}\n")
        arquivo.write(f"Total de registros invalidos: {len(dados_invalidos)}\n")
        arquivo.write(f"Media de idade: {estatisticas.get('media_idade', 0):.2f} anos\n")
        arquivo.write(f"Media de semestre: {estatisticas.get('media_semestre', 0):.2f}\n")
        arquivo.write(f"Media de horas de sono: {formatar_horas(estatisticas.get('media_horas_sono', 0))}\n")
        arquivo.write(f"Media de horas de estudo: {formatar_horas(estatisticas.get('media_horas_estudo', 0))}\n")
        arquivo.write(f"Media de horas em redes sociais: {formatar_horas(estatisticas.get('media_horas_redes', 0))}\n")
        arquivo.write(f"Mediana de horas de estudo: {formatar_horas(estatisticas.get('mediana_horas_estudo', 0))}\n")
        arquivo.write(f"Maior carga de estudo encontrada: {formatar_horas(estatisticas.get('max_horas_estudo', 0))}\n")
        arquivo.write(f"Menor carga de sono encontrada: {formatar_horas(estatisticas.get('min_horas_sono', 0))}\n")
        arquivo.write(f"Percentual que estudam no fim de semana: {estatisticas.get('percentual_estuda_fds', 0):.2f}%\n")
        arquivo.write(f"Percentual que trabalham: {estatisticas.get('percentual_trabalham', 0):.2f}%\n")
        arquivo.write(f"Percentual que bebem cafe: {estatisticas.get('percentual_bebem_cafe', 0):.2f}%\n")
        arquivo.write(f"Percentual com produtividade positiva: {estatisticas.get('percentual_produtividade_positiva', 0):.2f}%\n")
        arquivo.write(f"Percentual com menos de 6h de sono: {estatisticas.get('percentual_sono_abaixo_6h', 0):.2f}%\n")
        arquivo.write(f"Percentual com 4h ou mais em redes sociais: {estatisticas.get('percentual_redes_acima_4h', 0):.2f}%\n")
        arquivo.write(f"Media de estudo de quem trabalha: {formatar_horas(estatisticas.get('media_estudo_quem_trabalha', 0))}\n")
        arquivo.write(f"Media de estudo de quem nao trabalha: {formatar_horas(estatisticas.get('media_estudo_quem_nao_trabalha', 0))}\n")
        arquivo.write(f"Media de estudo de quem estuda no fim de semana: {formatar_horas(estatisticas.get('media_estudo_fim_semana', 0))}\n")
        arquivo.write(f"Razao media estudo/redes: {estatisticas.get('razao_estudo_redes', 0):.2f}\n")
        arquivo.write(f"Turno com maior media de estudo: {estatisticas.get('turno_maior_media_estudo', 'N/A')} ({formatar_horas(estatisticas.get('valor_turno_maior_media_estudo', 0))})\n")
        arquivo.write(f"Area com maior media de estudo: {estatisticas.get('area_maior_media_estudo', 'N/A')} ({formatar_horas(estatisticas.get('valor_area_maior_media_estudo', 0))})\n")
        arquivo.write(f"Produtividade mais comum: {estatisticas.get('moda_produtividade', 'N/A')}\n")
        arquivo.write(f"Turno mais comum: {estatisticas.get('moda_turno', 'N/A')}\n")
        arquivo.write(f"Total de areas unicas: {estatisticas.get('total_areas_unicas', 0)}\n")
        arquivo.write(f"Requisito minimo de 40 registros validos atendido: {'sim' if estatisticas.get('atingiu_minimo_registros') else 'nao'}\n\n")

        arquivo.write('ANALISES TEXTUAIS:\n')
        arquivo.write('-' * 40 + '\n')
        arquivo.write(f"Area mais frequente: {analises_textuais.get('area_mais_frequente', 'N/A')}\n")
        arquivo.write(f"Dificuldade mais citada: {analises_textuais.get('dificuldade_mais_citada', 'N/A')}\n")
        arquivo.write(f"Local de estudo mais comum: {analises_textuais.get('local_estudo_mais_comum', 'N/A')}\n")
        arquivo.write(f"Areas identificadas: {analises_textuais.get('areas_identificadas', 'N/A')}\n\n")

        arquivo.write('RANKING - MAIORES CARGAS HORARIAS DE ESTUDO:\n')
        arquivo.write('-' * 40 + '\n')
        for indice, item in enumerate(ranking_estudo[:10], start=1):
            area, turno, valor = item
            arquivo.write(f"{indice}. {formatar_horas(valor)} - Area: {area} - Turno: {turno}\n")
        arquivo.write('\n')

        arquivo.write('RANKING - MAIORES TEMPOS EM REDES SOCIAIS:\n')
        arquivo.write('-' * 40 + '\n')
        for indice, item in enumerate(ranking_redes[:10], start=1):
            area, turno, valor = item
            arquivo.write(f"{indice}. {formatar_horas(valor)} - Area: {area} - Turno: {turno}\n")
        arquivo.write('\n')

        if dados_invalidos:
            arquivo.write('REGISTROS INVALIDOS ENCONTRADOS:\n')
            arquivo.write('-' * 40 + '\n')
            for registro in dados_invalidos:
                erros = registro.get('erros_validacao')
                if isinstance(erros, str):
                    erros = [erros]
                elif not erros:
                    _, erros = validar_registro(registro)

                arquivo.write(f"Registro: {registro.get('timestamp', 'Sem data')}\n")
                for erro in erros:
                    arquivo.write(f"  - {erro}\n")
                arquivo.write('\n')

    print(f"Relatorio salvo em: {nome_arquivo}")


def salvar_csv(nome_arquivo, registros):
    """
    Salva uma lista de dicionarios em CSV separado por virgula.
    """
    if not registros:
        print(f"Nenhum registro para salvar em: {nome_arquivo}")
        return

    campos = list(registros[0].keys())
    for campo_extra in ['status_validacao', 'erros_validacao']:
        if any(campo_extra in registro for registro in registros) and campo_extra not in campos:
            campos.append(campo_extra)

    linhas_saida = []
    for registro in registros:
        linha = dict(registro)
        if isinstance(linha.get('erros_validacao'), list):
            linha['erros_validacao'] = ' | '.join(linha['erros_validacao'])
        linhas_saida.append(linha)

    with open(nome_arquivo, 'w', encoding='utf-8-sig', newline='') as arquivo:
        escritor = csv.DictWriter(arquivo, fieldnames=campos)
        escritor.writeheader()
        escritor.writerows(linhas_saida)

    print(f"Arquivo salvo em: {nome_arquivo}")


### `menu_principal()`
Serve para mostrar o menu interativo e capturar a opcao escolhida pelo usuario.


In [ ]:
def menu_principal():
    """
    Menu interativo principal do sistema.
    """
    print("\n\n")
    print("=" * 60)
    print('SISTEMA DE AUDITORIA E ANALISE DE DADOS - HABITOS DE ESTUDO')
    print('=' * 60)
    print('1. Carregar arquivo de dados')
    print('2. Validar registros e auditar duplicados')
    print('3. Mostrar estatisticas e analises')
    print('4. Filtrar registros por campo textual')
    print('   Ex.: turno, trabalha, produtividade, area_curso')
    print('5. Filtrar registros por intervalo numerico')
    print('   Ex.: idade, semestre, horas_sono, horas_estudo')
    print('6. Buscar por termo livre')
    print('   Ex.: tempo, casa, tecnologia, noturno')
    print('7. Gerar rankings')
    print('8. Salvar relatorio e arquivos CSV')
    print('9. Sair')
    print('-' * 60)

    opcao = input('Escolha uma opcao (1-9): ')
    return opcao


### Fluxo principal do sistema
- `carregar_dados_upload()` recebe o arquivo enviado no Colab.
- `executar_validacao()` organiza a validacao, a auditoria de duplicados e o calculo das analises.
- `possui_registros_validos()` evita executar opcoes sem dados validados.
- `exibir_estatisticas_e_analises()` apresenta os resultados na tela.
- `executar_filtro_textual()`, `executar_filtro_intervalo()` e `executar_busca()` tratam as consultas do usuario.
- `exibir_rankings()` mostra os rankings finais.
- `executar_exportacao()` salva relatorio e CSVs.
- `main()` coordena todo o fluxo do menu.


In [ ]:
def carregar_dados_upload():
    """
    Carrega o arquivo enviado no ambiente do Colab.
    """
    if 'uploaded' not in globals() or not uploaded:
        print('Execute a celula de upload antes de carregar o arquivo.')
        return []

    nome_arquivo = list(uploaded.keys())[0]
    print(f"\nArquivo carregado: {nome_arquivo}\n")
    dados = carregar_arquivo(nome_arquivo)

    if dados:
        print(f"Arquivo carregado com sucesso! {len(dados)} registros encontrados.")
    else:
        print('Falha ao carregar o arquivo.')

    return dados


def executar_validacao(dados_originais):
    """
    Executa validacao, auditoria de duplicados e calculo das analises.
    """
    dados_validos = []
    dados_invalidos = []
    dados_para_validar = []

    for registro in dados_originais:
        if registro.get('status_validacao') == 'invalido' and registro.get('erros_validacao'):
            dados_invalidos.append(registro)
        else:
            dados_para_validar.append(registro)

    dados_unicos, dados_duplicados = remover_duplicados(dados_para_validar)
    dados_invalidos.extend(dados_duplicados)

    for registro in dados_unicos:
        valido, erros = validar_registro(registro)
        if valido:
            dados_validos.append(registro)
        else:
            registro_invalido = dict(registro)
            registro_invalido['status_validacao'] = 'invalido'
            registro_invalido['erros_validacao'] = erros
            dados_invalidos.append(registro_invalido)

    estatisticas = gerar_estatisticas(dados_validos)
    estatisticas['total_invalidos'] = len(dados_invalidos)
    estatisticas['total_duplicados'] = len(dados_duplicados)
    analises_textuais = gerar_analises_textuais(dados_validos)
    ranking_estudo = gerar_ranking(dados_validos, 'horas_estudo', 'desc')
    ranking_redes = gerar_ranking(dados_validos, 'horas_redes', 'desc')

    print('\nValidacao concluida!')
    print(f"Registros carregados: {len(dados_originais)}")
    print(f"Registros unicos analisados: {len(dados_unicos)}")
    print(f"Registros validos: {len(dados_validos)}")
    print(f"Registros invalidos: {len(dados_invalidos)}")
    print(f"Duplicados identificados: {len(dados_duplicados)}")
    print('Requisito minimo de 40 registros validos atendido:' + (' sim' if estatisticas.get('atingiu_minimo_registros') else ' nao'))

    return {
        'dados_validos': dados_validos,
        'dados_invalidos': dados_invalidos,
        'dados_duplicados': dados_duplicados,
        'estatisticas': estatisticas,
        'analises_textuais': analises_textuais,
        'ranking_estudo': ranking_estudo,
        'ranking_redes': ranking_redes
    }


def possui_registros_validos(dados_validos):
    """
    Verifica se a validacao ja foi executada com sucesso.
    """
    if not dados_validos:
        print('Erro: valide os registros primeiro (opcao 2).')
        return False
    return True


def exibir_estatisticas_e_analises(estatisticas, analises_textuais):
    """
    Mostra estatisticas e analises textuais na tela.
    """
    linhas_estatisticas = [
        ('Total de registros validos', estatisticas['total_registros']),
        ('Media de idade', f"{estatisticas['media_idade']:.2f} anos"),
        ('Media de semestre', f"{estatisticas['media_semestre']:.2f}"),
        ('Media de horas de sono', formatar_horas(estatisticas['media_horas_sono'])),
        ('Media de horas de estudo', formatar_horas(estatisticas['media_horas_estudo'])),
        ('Media de horas em redes sociais', formatar_horas(estatisticas['media_horas_redes'])),
        ('Mediana de horas de estudo', formatar_horas(estatisticas['mediana_horas_estudo'])),
        ('Maior carga de estudo', formatar_horas(estatisticas['max_horas_estudo'])),
        ('Menor carga de sono', formatar_horas(estatisticas['min_horas_sono'])),
        ('Percentual que estudam no fim de semana', f"{estatisticas['percentual_estuda_fds']:.2f}%"),
        ('Percentual que trabalham', f"{estatisticas['percentual_trabalham']:.2f}%"),
        ('Percentual que bebem cafe', f"{estatisticas['percentual_bebem_cafe']:.2f}%"),
        ('Percentual com produtividade positiva', f"{estatisticas['percentual_produtividade_positiva']:.2f}%"),
        ('Percentual com menos de 6h de sono', f"{estatisticas['percentual_sono_abaixo_6h']:.2f}%"),
        ('Percentual com 4h ou mais em redes sociais', f"{estatisticas['percentual_redes_acima_4h']:.2f}%"),
        ('Media de estudo de quem trabalha', formatar_horas(estatisticas['media_estudo_quem_trabalha'])),
        ('Media de estudo de quem nao trabalha', formatar_horas(estatisticas['media_estudo_quem_nao_trabalha'])),
        ('Media de estudo de quem estuda no fim de semana', formatar_horas(estatisticas['media_estudo_fim_semana'])),
        ('Razao media estudo/redes', f"{estatisticas['razao_estudo_redes']:.2f}"),
        ('Turno com maior media de estudo', f"{estatisticas['turno_maior_media_estudo']} ({formatar_horas(estatisticas['valor_turno_maior_media_estudo'])})"),
        ('Area com maior media de estudo', f"{estatisticas['area_maior_media_estudo']} ({formatar_horas(estatisticas['valor_area_maior_media_estudo'])})"),
        ('Produtividade mais comum', estatisticas['moda_produtividade']),
        ('Turno mais comum', estatisticas['moda_turno']),
        ('Total de areas/cursos', estatisticas['total_areas']),
        ('Requisito minimo de 40 registros validos atendido', 'sim' if estatisticas['atingiu_minimo_registros'] else 'nao')
    ]

    linhas_analises = [
        ('Area mais frequente', analises_textuais['area_mais_frequente']),
        ('Dificuldade mais citada', analises_textuais['dificuldade_mais_citada']),
        ('Local de estudo mais comum', analises_textuais['local_estudo_mais_comum']),
        ('Areas identificadas', analises_textuais['areas_identificadas'])
    ]

    print('\n' + '=' * 50)
    print('ESTATISTICAS GERAIS')
    print('=' * 50)
    for titulo, valor in linhas_estatisticas:
        print(f"{titulo}: {valor}")
        print()

    print('\nANALISES TEXTUAIS')
    print('=' * 50)
    for titulo, valor in linhas_analises:
        print(f"{titulo}: {valor}")
        print()


def executar_filtro_textual(dados_validos):
    """
    Executa o filtro textual interativo.
    """
    exemplos_por_campo = {
        'area_curso': 'Exemplos: TI / Tecnologia, Saude, Engenharias, Juridicas',
        'turno': 'Valores aceitos: matutino, vespertino, noturno, integral',
        'produtividade': 'Valores aceitos: muito bom, bom, normal, ruim, muito ruim',
        'dificuldade': 'Exemplos: tempo, foco, concentracao, calculo',
        'lugar_estudo': 'Exemplos: casa, biblioteca, faculdade, outros',
        'trabalha': 'Valores aceitos: sim ou nao',
        'bebe_cafe': 'Valores aceitos: sim ou nao'
    }
    campos_busca_exata = {'turno', 'produtividade', 'trabalha', 'bebe_cafe'}

    print('\nFILTRO TEXTUAL')
    print('-' * 50)
    print('Voce vai escolher um campo e depois informar o valor que deseja filtrar.')
    print('Campos disponiveis:')
    for campo, exemplo in exemplos_por_campo.items():
        print(f'- {campo}: {exemplo}')

    campo = normalizar_texto(input('\nDigite o campo para filtrar: '))
    if campo not in exemplos_por_campo:
        print('Campo invalido. Escolha um dos campos listados.')
        return

    print(f"\nCampo escolhido: {campo}")
    print(exemplos_por_campo[campo])

    exato = campo in campos_busca_exata
    if exato:
        print('Esse campo usa comparacao exata automaticamente.')
    else:
        print('Esse campo usa busca parcial automaticamente.')

    valor = input('Digite o valor que deseja buscar: ')
    resultados = filtrar_registros(dados_validos, campo, valor, exato)

    print(f"\nEncontrados {len(resultados)} registros.")
    for indice, registro in enumerate(resultados[:20], start=1):
        print(f"{indice}. {registro.get('area_curso', 'N/A')} - Turno: {registro.get('turno', 'N/A')} - Dificuldade: {registro.get('dificuldade', 'N/A')}")


def executar_filtro_intervalo(dados_validos):
    """
    Executa o filtro numerico interativo.
    """
    exemplos_por_campo = {
        'idade': 'Exemplo de faixa: 20 ate 30',
        'semestre': 'Exemplo de faixa: 1 ate 6',
        'horas_sono': 'Exemplo de faixa: 05:00 ate 08:00',
        'horas_estudo': 'Exemplo de faixa: 02:00 ate 06:00',
        'horas_redes': 'Exemplo de faixa: 00:30 ate 04:00'
    }

    print('\nFILTRO NUMERICO')
    print('-' * 50)
    print('Voce vai escolher um campo numerico e informar um minimo e/ou um maximo.')
    print('Se nao quiser usar um dos limites, basta apertar Enter.')
    print('Campos disponiveis:')
    for campo, exemplo in exemplos_por_campo.items():
        print(f'- {campo}: {exemplo}')

    campo = normalizar_texto(input('\nDigite o campo: '))
    if campo not in exemplos_por_campo:
        print('Campo invalido. Escolha um dos campos listados.')
        return

    print(exemplos_por_campo[campo])
    minimo = input('Valor minimo (pressione Enter para ignorar): ')
    maximo = input('Valor maximo (pressione Enter para ignorar): ')

    resultados = filtrar_por_intervalo(dados_validos, campo, minimo, maximo)
    print(f"\nEncontrados {len(resultados)} registros no intervalo informado.")

    for indice, registro in enumerate(resultados[:20], start=1):
        print(f"{indice}. {registro.get('area_curso', 'N/A')} - {campo}: {registro.get(campo, 'N/A')} - Produtividade: {registro.get('produtividade', 'N/A')}")


def executar_busca(dados_validos):
    """
    Executa a busca textual livre na base.
    """
    print('\nBUSCA LIVRE')
    print('-' * 50)
    print('Essa opcao procura o termo em varios campos ao mesmo tempo.')
    print('Exemplos do que voce pode buscar: tempo, casa, tecnologia, noturno, cafe')

    termo = input('Digite a palavra ou expressao para buscar: ')
    resultados = buscar_por_termo(dados_validos, termo)
    print(f"\nEncontrados {len(resultados)} registros contendo '{termo}'.")

    for indice, registro in enumerate(resultados[:20], start=1):
        print(f"{indice}. {registro.get('area_curso', 'N/A')} - Local: {registro.get('lugar_estudo', 'N/A')} - Dificuldade: {registro.get('dificuldade', 'N/A')}")


def exibir_rankings(ranking_estudo, ranking_redes):
    """
    Mostra os rankings de estudo e redes sociais.
    """
    print('\nRANKING - MAIORES CARGAS HORARIAS DE ESTUDO:')
    print('-' * 50)
    for indice, item in enumerate(ranking_estudo[:10], start=1):
        area, turno, valor = item
        print(f"{indice}. {formatar_horas(valor)} - {area} - Turno: {turno}")

    print('\nRANKING - MAIORES TEMPOS EM REDES SOCIAIS:')
    print('-' * 50)
    for indice, item in enumerate(ranking_redes[:10], start=1):
        area, turno, valor = item
        print(f"{indice}. {formatar_horas(valor)} - {area} - Turno: {turno}")


def executar_exportacao(dados_validos, dados_invalidos, estatisticas, analises_textuais, ranking_estudo, ranking_redes):
    """
    Salva relatorio e arquivos auxiliares.
    """
    nome_relatorio = input('Digite o nome do arquivo de relatorio (ex: relatorio.txt): ')
    salvar_relatorio(
        dados_validos,
        dados_invalidos,
        estatisticas,
        analises_textuais,
        ranking_estudo,
        ranking_redes,
        nome_relatorio
    )
    salvar_csv('registros_validos.csv', dados_validos)
    salvar_csv('registros_invalidos.csv', dados_invalidos)


def main():
    """
    Funcao principal que orquestra o sistema.
    """
    dados_originais = []
    dados_validos = []
    dados_invalidos = []
    dados_duplicados = []
    estatisticas = {}
    analises_textuais = {}
    ranking_estudo = []
    ranking_redes = []

    while True:
        opcao = menu_principal()

        if opcao == '1':
            dados_originais = carregar_dados_upload()

        elif opcao == '2':
            if not dados_originais:
                print('Erro: carregue um arquivo primeiro (opcao 1).')
                continue

            resultado_validacao = executar_validacao(dados_originais)
            dados_validos = resultado_validacao['dados_validos']
            dados_invalidos = resultado_validacao['dados_invalidos']
            dados_duplicados = resultado_validacao['dados_duplicados']
            estatisticas = resultado_validacao['estatisticas']
            analises_textuais = resultado_validacao['analises_textuais']
            ranking_estudo = resultado_validacao['ranking_estudo']
            ranking_redes = resultado_validacao['ranking_redes']

        elif opcao == '3':
            if not possui_registros_validos(dados_validos):
                continue
            exibir_estatisticas_e_analises(estatisticas, analises_textuais)

        elif opcao == '4':
            if not possui_registros_validos(dados_validos):
                continue
            executar_filtro_textual(dados_validos)

        elif opcao == '5':
            if not possui_registros_validos(dados_validos):
                continue
            executar_filtro_intervalo(dados_validos)

        elif opcao == '6':
            if not possui_registros_validos(dados_validos):
                continue
            executar_busca(dados_validos)

        elif opcao == '7':
            if not possui_registros_validos(dados_validos):
                continue
            exibir_rankings(ranking_estudo, ranking_redes)

        elif opcao == '8':
            if not possui_registros_validos(dados_validos):
                continue
            executar_exportacao(
                dados_validos,
                dados_invalidos,
                estatisticas,
                analises_textuais,
                ranking_estudo,
                ranking_redes
            )

        elif opcao == '9':
            print('Encerrando o sistema...')
            break

        else:
            print('Opcao invalida! Escolha um numero de 1 a 9.')


# Execucao do sistema
Esta celula chama `main()` e inicia o menu interativo do projeto.


In [ ]:
main()




SISTEMA DE AUDITORIA E ANALISE DE DADOS - HABITOS DE ESTUDO
1. Carregar arquivo de dados
2. Validar registros e auditar duplicados
3. Mostrar estatisticas e analises
4. Filtrar registros por campo textual
   Ex.: turno, trabalha, produtividade, area_curso
5. Filtrar registros por intervalo numerico
   Ex.: idade, semestre, horas_sono, horas_estudo
6. Buscar por termo livre
   Ex.: tempo, casa, tecnologia, noturno
7. Gerar rankings
8. Salvar relatorio e arquivos CSV
9. Sair
------------------------------------------------------------
Erro: valide os registros primeiro (opcao 2).



SISTEMA DE AUDITORIA E ANALISE DE DADOS - HABITOS DE ESTUDO
1. Carregar arquivo de dados
2. Validar registros e auditar duplicados
3. Mostrar estatisticas e analises
4. Filtrar registros por campo textual
   Ex.: turno, trabalha, produtividade, area_curso
5. Filtrar registros por intervalo numerico
   Ex.: idade, semestre, horas_sono, horas_estudo
6. Buscar por termo livre
   Ex.: tempo, casa, tecnologia, 

## Testes realizados
- Foi realizado teste de leitura do arquivo CSV oficial no Colab com a estrutura completa das 14 colunas esperadas.
- Na base oficial de 42 registros, o sistema identificou 35 registros validos, 7 invalidos e 0 duplicados.
- Os testes de validacao confirmaram o funcionamento das regras de idade, semestre, campos obrigatorios, campos booleanos e turnos permitidos.
- A validacao de horario foi ajustada para aceitar formatos como HH:MM e HH:MM:SS, rejeitando minutos e segundos fora do intervalo permitido.
- A regra de sono minimo invalida respostas menores que 1 hora.
- A regra de consistencia invalida casos em que a soma de sono, estudo e redes ultrapassa 24 horas no mesmo registro.
- O sistema tambem invalida valores extremos de estudo acima de 18 horas e de redes sociais acima de 12 horas.
- Linhas CSV com quantidade incorreta de colunas deixam de ser ignoradas e passam a ser registradas como invalidas na auditoria.
- Foram testados os dois filtros dinamicos, a busca textual, a geracao dos rankings e a exportacao do relatorio e dos arquivos CSV.
- A exportacao dos CSVs foi mantida no mesmo padrao de leitura, permitindo salvar e reabrir os arquivos no proprio sistema.


## Analise dos resultados
Na base oficial de 42 registros, foram encontrados 35 registros validos, 7 invalidos e nenhum duplicado. A area mais frequente foi TI / Tecnologia, a dificuldade mais citada foi tempo e o local de estudo mais comum foi casa.

Entre os indicadores numericos, a media de idade foi de 25,11 anos, a media de sono foi de 06:45, a media de estudo foi de 03:28 e a media de uso de redes sociais foi de 02:46. A mediana de estudo ficou em 03:00 e a produtividade positiva apareceu em 48,57% dos registros validos.

Os resultados tambem mostraram que 20,00% dos respondentes dormem menos de 6 horas e 28,57% passam 4 horas ou mais em redes sociais. O turno com maior media de estudo foi o integral, com 04:26, e a area com maior media de estudo foi Juridicas, com 07:50.

Mesmo sem atingir ainda o minimo de 40 registros validos, o sistema esta consistente para auditoria, filtros, rankings e recalculo automatico das analises quando novos registros forem adicionados.


## Conclusoes do grupo
O desenvolvimento do projeto permitiu aplicar, de forma integrada, leitura de arquivos, validacao de dados, tratamento de dados, organizacao em listas e dicionarios, uso de tuplas e conjuntos, filtros, rankings e geracao de relatorios e analises.

A base oficial reforcou que as regras de auditoria sao importantes para separar respostas coerentes de valores extremos ou inconsistentes com a rotina diaria. Ao mesmo tempo, o sistema foi mantido simples, com funcoes separadas.

Como proximos passos imediatos do grupo:
- Completar a base ate atingir pelo menos 40 registros validos.
- Revisar os resultados finais apos a ultima coleta.
- Manter o codigo simples e coerente com as exigencias da atividade.


## Limitacoes da solucao
- A qualidade da analise depende diretamente da consistencia da base coletada.
- Como o projeto nao permite bibliotecas externas, algumas visualizacoes e automacoes mais avancadas nao foram incluidas.
- O sistema depende do arquivo seguir a estrutura esperada de colunas e respostas.
- Mesmo com validacao, a interpretacao final ainda depende de leitura humana dos resultados.


## Justificativa das estruturas utilizadas
- Listas foram usadas para armazenar colecoes de registros e resultados intermediarios.
- Dicionarios representam cada resposta da base, facilitando acesso por nome de campo.
- Tuplas aparecem nos rankings finais e nas assinaturas de duplicidade.
- Conjuntos foram usados para identificar valores unicos e detectar registros duplicados.
- Booleanos aparecem nas regras de validacao e no indicador de atingimento do minimo de 40 registros validos.
- Lambda, map e filter foram aplicados em etapas reais de extracao, limpeza, estatisticas e filtragem.

## Referencias
- Material das aulas de Python da disciplina Novas Tecnologias.
- Documento orientador da atividade N1-AT1.
- Documentacao oficial do Google Colab.
- Documentacao oficial do Python.
- Material complementar utilizado pelo grupo para revisao de logica e sintaxe da linguagem.
